# Validation: analytic statevectors, and union-find vs scipy



## Part 1 — analytic statevector vs Qiskit

`z_feature_map`/`zz_feature_map` are each one layer of `H^⊗n` followed by a purely
diagonal (phase-only) block. `statevectors()` builds that diagonal directly from closed-form
expressions instead of simulating gates, then applies `H^⊗n` (a Walsh-Hadamard transform)
between repetitions. It should reproduce Qiskit's own circuit simulation exactly, up to
floating-point rounding.

In [1]:
import numpy as np
import pandas as pd
from qiskit.circuit.library import z_feature_map, zz_feature_map
from qiskit.quantum_info import Statevector


def _fwht_inplace(a):
    """Unnormalised Walsh-Hadamard transform along axis 1."""
    N, dim = a.shape
    h = 1
    while h < dim:
        v = a.reshape(N, -1, 2, h)
        x, yy = v[:, :, 0, :], v[:, :, 1, :]
        t = x - yy
        x += yy
        yy[...] = t
        h *= 2
    return a


def statevectors(X, kind, reps, entanglement='linear'):
    """Exact statevectors of qiskit's z_feature_map / zz_feature_map, vectorised over rows of X."""
    X = np.asarray(X, dtype=float)
    N, n = X.shape
    dim = 2 ** n
    if kind not in ('z', 'zz'):
        raise ValueError(kind)
    if kind == 'zz' and entanglement != 'linear':
        raise ValueError("only 'linear' entanglement is implemented")

    single = np.empty((n, N, 2), dtype=complex)
    single[:, :, 0] = 1.0
    single[:, :, 1] = np.exp(1j * 2.0 * X.T)

    if kind == 'zz':
        pair = np.ones((n - 1, N, 2, 2), dtype=complex)
        for i in range(n - 1):
            e = np.exp(1j * 2.0 * (X[:, i] - np.pi) * (X[:, i + 1] - np.pi))
            pair[i, :, 0, 1] = e
            pair[i, :, 1, 0] = e

    acc = single[n - 1]
    for i in range(n - 2, -1, -1):
        acc = acc.reshape(N, -1, 2)
        new = acc[:, :, :, None] * single[i][:, None, None, :]
        if kind == 'zz':
            new = new * pair[i][:, None, :, :]
        acc = new.reshape(N, -1)
    diag = acc

    state = diag * (dim ** -0.5)
    for _ in range(reps - 1):
        state = _fwht_inplace(state) * (dim ** -0.5)
        state *= diag
    return state

### Test cases

Sweeps qubit count (1 to 15, covering the smallest possible case up to the largest
used in the project), both feature maps, repetition counts 1-3, and 5 random input
vectors per configuration (so a config isn't marked correct by a single lucky draw).
Each case compares the full statevector *and* the resulting `|<a|b>|^2` kernel matrix
built from 6 points, which is the quantity actually used downstream.

In [2]:
rng = np.random.default_rng(0)
rows = []
worst_state_err = 0.0
worst_kernel_err = 0.0

for n_qubits in [1, 2, 3, 5, 8, 9, 10, 12, 15]:
    for kind in ['z', 'zz']:
        if kind == 'zz' and n_qubits < 2:
            continue  # ZZFeatureMap needs at least 2 qubits (it entangles pairs)
        for reps in [1, 2, 3]:
            fm = (zz_feature_map(n_qubits, reps=reps, entanglement='linear') if kind == 'zz'
                  else z_feature_map(n_qubits, reps=reps))
            for trial in range(5):
                X = rng.uniform(-np.pi, np.pi, size=(6, n_qubits))
                ref = np.array([Statevector.from_instruction(fm.assign_parameters(x)).data for x in X])
                got = statevectors(X, kind, reps)

                state_err = np.abs(ref - got).max()
                K_ref = np.abs(ref.conj() @ ref.T) ** 2
                K_got = np.abs(got.conj() @ got.T) ** 2
                kernel_err = np.abs(K_ref - K_got).max()

                worst_state_err = max(worst_state_err, state_err)
                worst_kernel_err = max(worst_kernel_err, kernel_err)
                rows.append({'n_qubits': n_qubits, 'map': kind, 'reps': reps, 'trial': trial,
                            'max|d_statevector|': state_err, 'max|d_kernel|': kernel_err})

results = pd.DataFrame(rows)
n_cases = len(results)
n_pass = (results['max|d_statevector|'] < 1e-10).sum()
print(f'{n_cases} test cases run (9 qubit counts x up to 2 maps x 3 reps x 5 random trials, ZZ skipped at n_qubits=1)')
print(f'{n_pass}/{n_cases} passed at tolerance 1e-10')
print(f'worst statevector deviation overall: {worst_state_err:.3e}')
print(f'worst kernel deviation overall:      {worst_kernel_err:.3e}')
assert n_pass == n_cases, 'at least one configuration disagreed with Qiskit'
print('ALL CASES PASSED')

255 test cases run (9 qubit counts x up to 2 maps x 3 reps x 5 random trials, ZZ skipped at n_qubits=1)
255/255 passed at tolerance 1e-10
worst statevector deviation overall: 1.370e-15
worst kernel deviation overall:      1.799e-14
ALL CASES PASSED


In [3]:
summary = results.groupby(['map', 'reps', 'n_qubits'])['max|d_statevector|'].max().unstack('n_qubits')
summary

## Part 2 — union-find vs `scipy.sparse.csgraph.connected_components`



In [4]:
def build_groups_unionfind(n, edges):
    parent = list(range(n))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for i, j in edges:
        union(i, j)

    raw = [find(i) for i in range(n)]
    remap = {g: k for k, g in enumerate(sorted(set(raw)))}
    return np.array([remap[g] for g in raw])


def partitions_equal(labels_a, labels_b):
    """True iff the two labelings induce the SAME grouping of indices, ignoring
    which numeric id each group happens to be called (union-find and scipy will
    generally number groups differently even when the partition is identical)."""
    n = len(labels_a)
    for i in range(n):
        for j in range(i + 1, n):
            if (labels_a[i] == labels_a[j]) != (labels_b[i] == labels_b[j]):
                return False
    return True

### Synthetic test cases

Small graphs built by hand, each targeting a specific situation the grouping code has
to get right.

In [5]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components


def scipy_components(n, edges):
    if not edges:
        return np.arange(n)  # scipy needs at least one entry; no edges = all singletons
    rows_ = [e[0] for e in edges]
    cols_ = [e[1] for e in edges]
    adj = coo_matrix((np.ones(len(edges)), (rows_, cols_)), shape=(n, n))
    _, labels = connected_components(adj, directed=False)
    return labels


test_cases = {
    'chain (non-transitive: 0-1, 1-2, 2-3 -> one group of 4)':
        (4, [(0, 1), (1, 2), (2, 3)]),
    'two disconnected triangles':
        (6, [(0, 1), (1, 2), (0, 2), (3, 4), (4, 5), (3, 5)]),
    'all singletons (no edges)':
        (5, []),
    'complete graph (single group)':
        (5, [(i, j) for i in range(5) for j in range(i + 1, 5)]),
    'star (one hub connected to 4 leaves)':
        (5, [(0, 1), (0, 2), (0, 3), (0, 4)]),
    'one triangle plus two isolated singletons':
        (5, [(0, 1), (1, 2), (0, 2)]),
    'long chain of 10 nodes (deep transitive closure)':
        (10, [(i, i + 1) for i in range(9)]),
    'two chains that should stay separate':
        (8, [(0, 1), (1, 2), (2, 3), (4, 5), (5, 6), (6, 7)]),
}

rows = []
for name, (n, edges) in test_cases.items():
    uf = build_groups_unionfind(n, edges)
    sp = scipy_components(n, edges)
    match = partitions_equal(uf, sp)
    rows.append({'case': name, 'n_nodes': n, 'n_edges': len(edges),
                'n_groups_unionfind': len(np.unique(uf)),
                'n_groups_scipy': len(np.unique(sp)), 'partitions_match': match})

synthetic_results = pd.DataFrame(rows)
synthetic_results

In [6]:
assert synthetic_results['partitions_match'].all(), 'union-find disagreed with scipy on a synthetic case'
print(f"ALL {len(synthetic_results)} SYNTHETIC CASES PASSED")

ALL 8 SYNTHETIC CASES PASSED


### Real data: the 80 peptide sequences

In [7]:
_lab = pd.read_excel('Docking_high_low_energy_labels.xlsx', sheet_name='Results')
_lab = _lab.dropna(subset=['Sequence Epitope)']).reset_index(drop=True)
sequences = _lab['Sequence Epitope)'].astype(str).str.strip().tolist()

K_SHARED, IDENTITY_THRESH = 5, 7 / 9


def similarity_edges(sequences, k_shared=K_SHARED, identity_thresh=IDENTITY_THRESH):
    n, L = len(sequences), len(sequences[0])
    kmers = [{s[i:i + k_shared] for i in range(L - k_shared + 1)} for s in sequences]
    edges = []
    for i in range(n):
        for j in range(i + 1, n):
            shared = bool(kmers[i] & kmers[j])
            identity = sum(a == b for a, b in zip(sequences[i], sequences[j])) / L
            if shared or identity >= identity_thresh:
                edges.append((i, j))
    return edges


edges = similarity_edges(sequences)
n_pep = len(sequences)

uf_groups = build_groups_unionfind(n_pep, edges)
sp_groups = scipy_components(n_pep, edges)

match = partitions_equal(uf_groups, sp_groups)
n_groups = len(np.unique(uf_groups))
n_singletons = int(pd.Series(uf_groups).value_counts().eq(1).sum())

print(f'{n_pep} peptides, {len(edges)} similarity edges')
print(f'union-find groups: {n_groups}, scipy groups: {len(np.unique(sp_groups))}')
print(f'singleton groups: {n_singletons}')
print(f'partitions match: {match}')
assert match, 'union-find and scipy disagree on the real dataset'
print('REAL-DATA CASE PASSED')

80 peptides, 56 similarity edges
union-find groups: 43, scipy groups: 43
singleton groups: 25
partitions match: True
REAL-DATA CASE PASSED


### Evidence that transitive closure was actually needed, not just a precaution



In [8]:
import itertools

edge_set = set(edges)
non_clique_groups = 0
for g in np.unique(uf_groups):
    members = np.where(uf_groups == g)[0]
    if len(members) > 2:
        for a, b in itertools.combinations(members, 2):
            if (min(a, b), max(a, b)) not in edge_set:
                non_clique_groups += 1
                break

print(f'groups held together only by transitive closure: {non_clique_groups} '
      f'out of {n_groups}')
if non_clique_groups > 0:
    print('-> the non-transitivity of the pairwise rule was a real issue in this '
         'dataset: these groups would have been split incorrectly without the '
         'connected-components treatment.')

groups held together only by transitive closure: 5 out of 43
-> the non-transitivity of the pairwise rule was a real issue in this dataset: these groups would have been split incorrectly without the connected-components treatment.
